# Day 20: Multi-Step Planning

Walkthrough of plan structure and validation, execution in dependency order, replanning after a failure, planning vs ReAct, step budgets, parallel levels, and the full pipeline end to end. Everything here runs offline with a scripted fake planner, no API key required.

## 1. What is multi-step planning

Write the whole plan first, then execute it, instead of deciding one step at a time like Day 17's ReAct loop.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../day-15-agentic-ai'))
sys.path.insert(0, os.path.abspath('../day-16-custom-tools'))
sys.path.insert(0, os.path.abspath('../day-17-react-pattern'))
sys.path.insert(0, os.path.abspath('../day-19-function-calling'))
print("A plan is a list of steps, each with a tool, arguments, and what it depends on.")
print("A good travel agent writes an itinerary first, then adjusts it if a flight is cancelled.")

## 2. Plan structure and validation

A Step and a Plan dataclass, plus validate_plan() checking for unknown tools, unknown dependency ids, and cycles.

In [ ]:
from plan_structure import Plan, Step, validate_plan

known_tools = {"lookup_order", "check_refund_eligibility", "create_ticket"}

good_plan = Plan(goal="Handle damaged order", steps=[
    Step("s1", "Look up the order", "lookup_order", {"order_id": "ORD-1001"}),
    Step("s2", "Check refund eligibility", "check_refund_eligibility", {"order_id": "ORD-1001"}, depends_on=["s1"]),
])
print(validate_plan(good_plan, known_tools))

cyclic_plan = Plan(goal="Broken", steps=[
    Step("s1", "A", "lookup_order", {"order_id": "ORD-1001"}, depends_on=["s2"]),
    Step("s2", "B", "check_refund_eligibility", {"order_id": "ORD-1001"}, depends_on=["s1"]),
])
print(validate_plan(cyclic_plan, known_tools))

## 3. Execute a plan

Running steps in dependency order, with one step's real output feeding a later step's arguments.

In [ ]:
from fake_planner import fake_plan
from plan_executor import execute_plan

plan = fake_plan("handle damaged order 1001")
results = execute_plan(plan)
print(f"\nStatuses: " + ", ".join(f"{sid}={r['status']}" for sid, r in results.items()))

## 4. Replanning after a failure

When a step fails, asking the fake planner for a revised plan, capped at a maximum number of replans.

In [ ]:
from replanning import run_with_replanning, report_outcome

print("### A failing attempt followed by a successful replan ###\n")
results = run_with_replanning(
    goal="handle damaged order 9999",
    revised_goal_for_failure="handle damaged order 9999 (revised after failure)",
)
report_outcome(results)

## 5. Planning vs ReAct

The same goal run both ways: ReAct deciding one step at a time, planning committing to a full plan up front.

In [ ]:
from plan_vs_react import run_react_version, run_planning_version

react_calls = run_react_version()
print("=" * 40)
planning_calls = run_planning_version()
print(f"\nReAct tool calls: {react_calls}, Planning tool calls: {planning_calls}")

## 6. Step budgets

Capping a plan at a maximum number of units, stopping and honestly reporting what was left undone once the budget runs out.

In [ ]:
from cost_and_step_budget import execute_plan_with_budget
from plan_structure import Plan, Step

plan = Plan(goal="Handle damaged order ORD-1001", steps=[
    Step("s1", "Look up the order", "lookup_order", {"order_id": "ORD-1001"}),
    Step("s2", "Check refund eligibility", "check_refund_eligibility", {"order_id": "ORD-1001"}, depends_on=["s1"]),
    Step("s3", "Create a ticket", "create_ticket", {"issue": "Damaged item", "priority": "high"}, depends_on=["s2"]),
])

results, spent = execute_plan_with_budget(plan, max_budget=2)
print(f"\nSpent {spent} of 2 units")

## 7. Parallel levels

Grouping independent steps into levels, showing which ones could run together and which have to wait for every dependency.

In [ ]:
from planning_with_dependencies import group_into_levels

parallel_plan = Plan(goal="Compare two orders", steps=[
    Step("s1", "Look up the first order", "lookup_order", {"order_id": "ORD-1001"}),
    Step("s2", "Look up the second order", "lookup_order", {"order_id": "ORD-1002"}),
    Step("s3", "Summarize", "create_ticket", {"issue": "Comparison", "priority": "low"}, depends_on=["s1", "s2"]),
])

levels = group_into_levels(parallel_plan.steps)
for i, level in enumerate(levels):
    print(f"Level {i}: {[s.step_id for s in level]}")

## 8. Full pipeline

Plan, validate, execute, replan if needed, and report honestly, all together.

In [ ]:
from agent_with_planning import handle_goal, print_final_report

outcome = handle_goal(
    goal="handle damaged order 9999",
    revised_goal_for_failure="handle damaged order 9999 (revised after failure)",
)
print_final_report(outcome)

## 9. Key takeaways

- A plan is a list of steps, each naming a tool, its arguments, and which earlier steps it depends on.
- Validating a plan before running it catches unknown tools, unknown dependency ids, and cycles as readable bugs instead of runtime crashes.
- Executing in dependency order means a failed step correctly skips everything downstream of it, instead of running on data that was never actually produced.
- Replanning is worth it when the failure itself suggests a different approach would work, and a replan cap keeps a planner that cannot fix the problem from trying forever.
- Planning lets the whole path be inspected and budgeted before anything runs. ReAct never commits past the next step, so it adapts to a surprise immediately, without needing to detect a failure and ask for a new plan first.
- A step budget turns an unbounded task into a known, predictable upper bound on tool calls, and reporting exactly what was cut keeps that cutoff honest.
- Independent steps can be grouped into levels and run together, the same idea Day 19 showed for parallel tool calls in a single turn, just applied across a whole plan's dependency graph.